Import các thư viện cần thiết

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin

** Phương pháp thu thập dữ liệu**

*   Sử dụng thư viện requests để gửi yêu cầu đến trang web.
*   Sử dụng BeautifulSoup để đọc và xử lý nội dung HTML.
*   Dùng biểu thức chính quy (Regex) để tách các thông tin như nhiệt độ, độ ẩm, áp suất và tốc độ gió.
*   Dữ liệu của các tỉnh/thành được gom vào một DataFrame bằng thư viện pandas.
*   Kết quả cuối cùng được lưu vào hai file weather_data.csv và weather_data.json.


In [13]:
BASE_URL = "https://xemthoitiet.vn/"
headers = {
    "User-Agent": "Mozilla/5.0"
}

BASE_URL là địa chỉ trang web dùng để thu thập dữ liệu thời tiết.

headers chứa thông tin User-Agent để request.

In [14]:
def get_locations():
    response = requests.get(BASE_URL, headers=headers)
    if response.status_code != 200:
        print("NOT FOUND!")
        return {}
    soup = BeautifulSoup(response.text, "html.parser")
    locations = {}
    for a in soup.find_all("a", href=True):
        href = a["href"]
        name = a.get_text(" ", strip=True)
        if re.fullmatch(r"/thoi-tiet/[^/]+/?", href):
            if (
                name != ""
                and "Hôm nay" not in name
                and "arrow_circle_right" not in name
                and "Thời tiết" not in name
                and len(name) < 30
            ):
                locations[name] = urljoin(BASE_URL, href)
    return locations

Hàm get_locations() dùng để truy cập trang chủ, tìm các đường dẫn thời tiết của từng tỉnh/thành và lọc bỏ các tên không hợp lệ. Kết quả trả về gồm tên địa điểm và URL.

In [29]:
def crawl_weather(location, base_url):
    url = base_url.rstrip("/") + "/7-ngay-toi/"
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return []
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    start = text.find("Dự báo thời tiết")
    end = text.find("Nhiệt độ và khả năng có mưa")
    if start == -1 or end == -1:
        return []

    weather_text = text[start:end]
    pattern = (
        r"(Hôm nay|T[2-7]\s+\d{2}/\d{2}|CN\s+\d{2}/\d{2})\s+"
        r"(\d+)°\s*/\s*(\d+)°\s+"
        r"(.+?)\s+(\d+)\s*%\s+([\d.]+)\s*km/h\s+"
        r"Ngày/đêm\s+(\d+)°/(\d+)°\s+"
        r"Sáng/tối\s+(\d+)°/(\d+)°\s+"
        r"Áp suất\s+(\d+)\s*hPa\s+"
        r"Mặt trời mọc lặn\s+"
        r"([0-9:]+\s*[ap]m)\s*/\s*([0-9:]+\s*[ap]m)\s+"
        r"Độ ẩm\s+(\d+)%\s+"
        r"Gió\s+([\d.]+)\s*km/h"
    )

    matches = re.findall(
        pattern,
        weather_text,
        re.IGNORECASE
    )

    data = []

    for item in matches:
        row = {
            "location": location,
            "date": item[0],
            "temp_min_c": int(item[1]),
            "temp_max_c": int(item[2]),
            "condition": item[3].strip(),
            "day_temp_c": int(item[6]),
            "night_temp_c": int(item[7]),
            "morning_temp_c": int(item[8]),
            "evening_temp_c": int(item[9]),
            "pressure_hpa": int(item[10]),
            "sunrise": item[11],
            "sunset": item[12],
            "humidity_percent": int(item[13]),
            "wind_speed_kmh": float(item[14])
        }
        data.append(row)
    return data

Hàm crawl_weather() dùng để truy cập trang dự báo thời tiết 7 ngày của từng địa điểm, lấy phần nội dung cần thiết. Dùng biểu thức chính quy pattern để tách các thông tin như ngày, nhiệt độ, tình trạng thời tiết, áp suất, độ ẩm, giờ mặt trời mọc/lặn và tốc độ gió.

Dùng re.findall() để tìm tất cả các phần trong weather_text khớp với mẫu pattern

Các thuộc tính:
1.   location: Tên tỉnh/ thành
2.   date: Ngày dự báo
3.   temp_min_c: Nhiệt độ thấp nhất
4.   temp_max_c: Nhiết độ cao nhất
5.   condition: Tình trạng thời tiết
6.   day_temp_c: Nhiệt độ ban ngày
7.   night_temp_c: Nhiệt độ ban đêm
8.   morning_temp_c: Nhiệt độ buổi sáng
9.   evening_temp_c: Nhiệt độ buổi tối
10.   pressure_hpa: Áp suất khí quyển
11.   sunrise: Thời gian mặt trời mọc
12.   sunset: Thời gian mặt trời lặn
13.   humidity_percent: Phần trăm độ ẩm
14.   wind_speed_kmh: Tốc độ gió

In [30]:
locations = get_locations()
print("Số địa điểm tìm được:", len(locations))
all_data = []
for name, url in locations.items():
    print("Đang lấy dữ liệu:", name)
    weather_data = crawl_weather(name, url)
    all_data.extend(weather_data)
    time.sleep(1)


Số địa điểm tìm được: 64
Đang lấy dữ liệu: Hà Giang
Đang lấy dữ liệu: Cao Bằng
Đang lấy dữ liệu: Bắc Kạn
Đang lấy dữ liệu: Tuyên Quang
Đang lấy dữ liệu: Thái Nguyên
Đang lấy dữ liệu: Lạng Sơn
Đang lấy dữ liệu: Quảng Ninh
Đang lấy dữ liệu: Bắc Giang
Đang lấy dữ liệu: Phú Thọ
Đang lấy dữ liệu: Lào Cai
Đang lấy dữ liệu: Điện Biên
Đang lấy dữ liệu: Lai Châu
Đang lấy dữ liệu: Sơn La
Đang lấy dữ liệu: Yên Bái
Đang lấy dữ liệu: Hòa Bình
Đang lấy dữ liệu: Hà Nội
Đang lấy dữ liệu: Vĩnh Phúc
Đang lấy dữ liệu: Bắc Ninh
Đang lấy dữ liệu: Hải Dương
Đang lấy dữ liệu: Hải Phòng
Đang lấy dữ liệu: Hưng Yên
Đang lấy dữ liệu: Thái Bình
Đang lấy dữ liệu: Hà Nam
Đang lấy dữ liệu: Nam Định
Đang lấy dữ liệu: Ninh Bình
Đang lấy dữ liệu: Thanh Hóa
Đang lấy dữ liệu: Nghệ An
Đang lấy dữ liệu: Hà Tĩnh
Đang lấy dữ liệu: Quảng Bình
Đang lấy dữ liệu: Quảng Trị
Đang lấy dữ liệu: Thừa Thiên Huế
Đang lấy dữ liệu: Đà Nẵng
Đang lấy dữ liệu: Quảng Nam
Đang lấy dữ liệu: Quảng Ngãi
Đang lấy dữ liệu: Bình Định
Đang lấy dữ li

get_locations() để lấy danh sách địa điểm, sau đó lần lượt duyệt từng tỉnh/thành và dùng crawl_weather() để thu thập dữ liệu thời tiết. Tất cả dữ liệu được gộp vào all_data, time.sleep(1) giúp tạo khoảng nghỉ 1 giây giữa các lần gửi yêu cầu đến website.

In [31]:
df = pd.DataFrame(all_data)
print(df.head())

if not df.empty:
    df.to_csv(
        "weather_data.csv",
        index=False,
        encoding="utf-8-sig"
    )
    df.to_json(
        "weather_data.json",
        orient="records",
        force_ascii=False,
        indent=4
    )
    print("DONE.")
else:
    print("ERROR")

   location      date  temp_min_c  temp_max_c            condition  \
0  Hà Giang   Hôm nay          19          29             Mây thưa   
1  Hà Giang  T4 07/10          17          31  Bầu trời quang đãng   
2  Hà Giang  T5 08/10          16          32  Bầu trời quang đãng   
3  Hà Giang  T6 09/10          16          32  Bầu trời quang đãng   
4  Hà Giang  T7 10/10          18          33             Mây thưa   

   day_temp_c  night_temp_c  morning_temp_c  evening_temp_c  pressure_hpa  \
0          29            19              20              23          1019   
1          31            17              19              23          1018   
2          32            16              19              25          1014   
3          32            19              20              26          1013   
4          33            18              21              27          1014   

   sunrise   sunset  humidity_percent  wind_speed_kmh  
0  5:53 am  5:43 pm                41            1.95  
1  5

Chuyển all_data thành DataFrame bằng Pandas và hiển thị 5 dòng đầu để kiểm tra dữ liệu. Nếu dữ liệu không rỗng thì lưu thành hai file CSV và JSON; ngược lại sẽ in "ERROR".